# Fragment-Level Analysis of Evaluated Sequences

This notebook loads the output CSV from `evaluate_sequences` and the fragment dictionary
from `sample_from_policy`, then aggregates sequence-level metrics down to fragment-level
metrics. For each unique fragment, we find all full-length sequences it appeared in and
average their metrics — giving a representative score for that fragment.

The user can then rank fragments per region, select top-k, and export them to FASTA for ordering.

In [ ]:
import json
import numpy as np
import pandas as pd

## 1. Load data

Set the paths to your evaluation CSV, fragment dictionary JSON, and the connector string
used when resampling fragments.

In [ ]:
# --- USER: set these paths ---
evaluation_csv_path = "PATH_TO_EVALUATION_RESULTS.csv"
fragment_dict_path = "PATH_TO_FRAGMENT_DICTIONARY.json"
connector = "___"
# ---

In [ ]:
seq_metrics = pd.read_csv(evaluation_csv_path)
print(f"Loaded {len(seq_metrics)} sequences with columns:")
print(list(seq_metrics.columns))

In [ ]:
with open(fragment_dict_path) as f:
    fragment_dict = json.load(f)

fragment_regions = sorted(fragment_dict.keys())
num_regions = len(fragment_regions)
print(f"Loaded {num_regions} fragment regions:")
for region in fragment_regions:
    print(f"  {region}: {len(fragment_dict[region])} unique fragments")

In [ ]:
# Build a lookup from fragment name -> sequence
frag_name_to_seq = {}
for region in fragment_regions:
    for name, seq in fragment_dict[region]:
        frag_name_to_seq[name] = seq

print(f"Total unique fragments across all regions: {len(frag_name_to_seq)}")

## 2. Parse fragment names from sequence names

Each sequence name in the evaluation CSV is the fragment names joined by the connector string.
We split them back out into per-region columns.

In [ ]:
frag_columns = [f"frag_{i+1}_name" for i in range(num_regions)]

split_names = seq_metrics["name"].str.split(connector, expand=True)
split_names.columns = frag_columns
seq_metrics = pd.concat([seq_metrics, split_names], axis=1)

seq_metrics[frag_columns].head()

## 3. Aggregate metrics from sequence level to fragment level

For each unique fragment, we find all sequences it appeared in and compute the mean and
standard deviation of every numeric metric column.

In [ ]:
# Identify numeric metric columns (exclude name and fragment name columns)
exclude_cols = {"name", "sequence"} | set(frag_columns)
metric_cols = [c for c in seq_metrics.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(seq_metrics[c])]
print(f"Aggregating {len(metric_cols)} metric columns: {metric_cols}")

In [ ]:
records = []

for region_idx, region_name in enumerate(fragment_regions):
    frag_col = frag_columns[region_idx]
    unique_frags = seq_metrics[frag_col].unique()

    for frag_name in unique_frags:
        subset = seq_metrics[seq_metrics[frag_col] == frag_name]

        record = {
            "frag_name": frag_name,
            "frag_seq": frag_name_to_seq.get(frag_name, ""),
            "frag_region": region_name,
            "frag_region_idx": region_idx + 1,
            "num_sequences": len(subset),
        }

        for col in metric_cols:
            record[f"{col}_mean"] = subset[col].mean()
            record[f"{col}_std"] = subset[col].std()

        records.append(record)

fragment_metrics = pd.DataFrame(records)
print(f"Aggregated metrics for {len(fragment_metrics)} fragments across {num_regions} regions")
fragment_metrics.head()

## 4. Rank fragments per region

Choose a metric to rank by and inspect the top fragments per region.
Adjust `rank_by_metric` and `rank_ascending` to control the ranking.

In [ ]:
# --- USER: set ranking parameters ---
rank_by_metric = "boltz_iptm_mean"  # column name to rank by
rank_ascending = False               # False = higher is better, True = lower is better
top_k = 20                           # number of top fragments to select per region
# ---

assert rank_by_metric in fragment_metrics.columns, (
    f"{rank_by_metric} not found. Available columns: {list(fragment_metrics.columns)}"
)

In [ ]:
top_fragments = {}

for region_name in fragment_regions:
    region_df = fragment_metrics[fragment_metrics["frag_region"] == region_name]
    ranked = region_df.sort_values(rank_by_metric, ascending=rank_ascending)
    top = ranked.head(top_k)
    top_fragments[region_name] = top

    print(f"\n--- {region_name} (top {len(top)}) ---")
    display_cols = ["frag_name", "frag_seq", "num_sequences", rank_by_metric]
    print(top[display_cols].to_string(index=False))

## 5. Export top fragments to FASTA

Write the selected top-k fragments to FASTA files — one per region, plus a combined file.
Set the output directory below.

In [ ]:
import os

# --- USER: set output directory ---
output_dir = "top_fragments_for_ordering"
# ---

os.makedirs(output_dir, exist_ok=True)

# Write per-region FASTA files
for region_name, top_df in top_fragments.items():
    fasta_path = os.path.join(output_dir, f"{region_name}_top{top_k}.fasta")
    with open(fasta_path, "w") as f:
        for _, row in top_df.iterrows():
            f.write(f">{row['frag_name']}\n{row['frag_seq']}\n")
    print(f"Wrote {len(top_df)} fragments to {fasta_path}")

# Write combined FASTA
combined_path = os.path.join(output_dir, f"all_regions_top{top_k}.fasta")
with open(combined_path, "w") as f:
    for region_name, top_df in top_fragments.items():
        for _, row in top_df.iterrows():
            f.write(f">{row['frag_name']}\n{row['frag_seq']}\n")
total = sum(len(df) for df in top_fragments.values())
print(f"\nWrote {total} total fragments to {combined_path}")

## 6. (Optional) Save full fragment metrics to CSV

In [ ]:
metrics_csv_path = os.path.join(output_dir, "fragment_metrics.csv")
fragment_metrics.to_csv(metrics_csv_path, index=False)
print(f"Saved full fragment metrics to {metrics_csv_path}")